# 🚦 GTSDB Traffic Sign Recognition — Google Colab GPU Training Notebook

**Author**: Nathenael Ermias  
**Dataset**: German Traffic Sign Detection Benchmark (GTSDB)  
**Architecture**: YOLOv8 (Ultralytics) • **Export Target**: ONNX Runtime & TFLite  
**GitHub Repository**: [https://github.com/Nathenael11/Traffic-Sign-Recognition](https://github.com/Nathenael11/Traffic-Sign-Recognition)  

---

## 📌 Notebook Overview & Colab Instructions
This notebook provides a complete deep learning workflow to train, evaluate, and export a YOLOv8 object detection model on the GTSDB dataset using **Google Colab GPU acceleration** (Tesla T4 / V100).

### Step-by-Step Instructions:
1. **Enable GPU**: Click `Runtime` -> `Change runtime type` -> Select `GPU (T4)`.
2. **Upload Dataset**: Upload `GTSDB_Train_and_Test.zip` or mount your Google Drive.
3. **Run All Cells**: Execute the notebook sequentially to train the model, evaluate held-out test accuracy, and export `best.onnx`.
4. **Update App**: Download `best.onnx` and `model_metrics.json` and place them in the `models/` directory of your web application repo!


## ⚙️ Step 1: Install Dependencies & Setup Environment


In [1]:
# Install Ultralytics YOLOv8, ONNX, and OpenCV dependencies
!pip install -q ultralytics onnx onnxruntime onnxslim opencv-python pillow pandas matplotlib

import os
import glob
import json
import time
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
from ultralytics import YOLO

print('PyTorch Version:', torch.__version__)
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 3.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
PyTorch Version: 2.11.0+cu128
GPU Available: True
GPU Device: Tesla T4


## 📂 Step 2: Upload or Mount GTSDB Dataset
If using Google Drive, mount drive below. Otherwise, upload `GTSDB_Train_and_Test.zip` to Colab root.


In [2]:
# Mount Google Drive if dataset is saved in Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Local/Colab standalone execution:', e)

# Extract zip automatically if present
if os.path.exists('/content/GTSDB_Train_and_Test.zip'):
    print('Extracting /content/GTSDB_Train_and_Test.zip...')
    !unzip -q -o /content/GTSDB_Train_and_Test.zip -d /content/

# Robust directory finder: Search /content for directory containing Train/images
def find_gtsdb_dir():
    for root, dirs, files in os.walk('/content'):
        train_img_path = os.path.join(root, 'Train', 'images')
        if os.path.exists(train_img_path) and len(glob.glob(os.path.join(train_img_path, '*.*'))) > 0:
            return root
    return '/content/GTSDB_Train_and_Test'

RAW_DATA_DIR = find_gtsdb_dir()
print('Detected GTSDB Raw Dataset Directory:', RAW_DATA_DIR)
train_count = len(glob.glob(os.path.join(RAW_DATA_DIR, 'Train', 'images', '*.*')))
test_count = len(glob.glob(os.path.join(RAW_DATA_DIR, 'Test', 'images', '*.*')))
print(f'Found {train_count} Train images and {test_count} Test images.')


Mounted at /content/drive
Detected GTSDB Raw Dataset Directory: /content/drive/MyDrive/traffic_sign_recognition/data
Found 600 Train images and 300 Test images.


## 📊 Step 3: Class Definitions & Data Imbalance Analysis
We list all 43 German Traffic Sign categories (index 0 to 42) and analyze per-class instance counts.


In [3]:
CLASS_NAMES = [
    'Speed limit (20km/h)', 'Speed limit (30km/h)', 'Speed limit (50km/h)', 'Speed limit (60km/h)',
    'Speed limit (70km/h)', 'Speed limit (80km/h)', 'End of speed limit (80km/h)', 'Speed limit (100km/h)',
    'Speed limit (120km/h)', 'No passing', 'No passing for heavy vehicles', 'Right-of-way at intersection',
    'Priority road', 'Yield', 'Stop', 'No vehicles', 'Heavy vehicles prohibited', 'No entry',
    'General caution', 'Dangerous curve left', 'Dangerous curve right', 'Double curve', 'Bumpy road',
    'Slippery road', 'Road narrows right', 'Road work', 'Traffic signals', 'Pedestrians',
    'Children crossing', 'Bicycles crossing', 'Beware of ice/snow', 'Wild animals crossing',
    'End of speed and passing limits', 'Turn right ahead', 'Turn left ahead', 'Ahead only',
    'Go straight or right', 'Go straight or left', 'Keep right', 'Keep left', 'Roundabout mandatory',
    'End of no passing', 'End of no passing for heavy vehicles'
]

# Count instances per class in raw Train labels
train_lbl_dir = os.path.join(RAW_DATA_DIR, 'Train', 'labels') if os.path.exists(RAW_DATA_DIR) else ''
class_train_counts = {i: 0 for i in range(43)}

if os.path.exists(train_lbl_dir):
    for lbl_file in glob.glob(os.path.join(train_lbl_dir, '*.txt')):
        with open(lbl_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts and parts[0].isdigit():
                    cid = int(parts[0])
                    if 0 <= cid < 43:
                        class_train_counts[cid] += 1

df_stats = pd.DataFrame([
    {'Class ID': i, 'Class Name': CLASS_NAMES[i], 'Train Instances': class_train_counts[i]}
    for i in range(43)
])
df_stats


,Class ID,Class Name,Train Instances
0,0,Speed limit (20km/h),3
1,1,Speed limit (30km/h),32
2,2,Speed limit (50km/h),43
3,3,Speed limit (60km/h),16
4,4,Speed limit (70km/h),15
5,5,Speed limit (80km/h),22
6,6,End of speed limit (80km/h),9
7,7,Speed limit (100km/h),23
8,8,Speed limit (120km/h),17
9,9,No passing,22


## 🛠️ Step 4: YOLO Format Dataset Structuring (85/15 Train/Val Split)


In [4]:
YOLO_DATA_DIR = '/content/gtsdb_yolo'
target_dirs = {
    'train_img': os.path.join(YOLO_DATA_DIR, 'images', 'train'),
    'train_lbl': os.path.join(YOLO_DATA_DIR, 'labels', 'train'),
    'val_img': os.path.join(YOLO_DATA_DIR, 'images', 'val'),
    'val_lbl': os.path.join(YOLO_DATA_DIR, 'labels', 'val'),
    'test_img': os.path.join(YOLO_DATA_DIR, 'images', 'test'),
    'test_lbl': os.path.join(YOLO_DATA_DIR, 'labels', 'test'),
}
for d in target_dirs.values():
    os.makedirs(d, exist_ok=True)

train_img_dir = os.path.join(RAW_DATA_DIR, 'Train', 'images')
train_lbl_dir = os.path.join(RAW_DATA_DIR, 'Train', 'labels')
test_img_dir = os.path.join(RAW_DATA_DIR, 'Test', 'images')
test_lbl_dir = os.path.join(RAW_DATA_DIR, 'Test', 'labels')

all_train_imgs = glob.glob(os.path.join(train_img_dir, '*.*'))
if len(all_train_imgs) == 0:
    raise FileNotFoundError(f'No training images found at {train_img_dir}! Please check Step 2 and upload GTSDB_Train_and_Test.zip.')

random.seed(42)
random.shuffle(all_train_imgs)

split_idx = int(len(all_train_imgs) * 0.85)
train_files = all_train_imgs[:split_idx]
val_files = all_train_imgs[split_idx:]

def copy_and_clean(file_list, dst_img, dst_lbl, src_lbl_dir):
    for img_path in file_list:
        base_name = os.path.basename(img_path)
        stem, _ = os.path.splitext(base_name)
        shutil.copy2(img_path, os.path.join(dst_img, base_name))
        lbl_src = os.path.join(src_lbl_dir, f'{stem}.txt')
        if os.path.exists(lbl_src):
            clean_lines = []
            with open(lbl_src, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts and parts[0].isdigit():
                        cid = int(parts[0])
                        if 0 <= cid < 43:
                            clean_lines.append(line.strip())
            with open(os.path.join(dst_lbl, f'{stem}.txt'), 'w') as f:
                f.write('\n'.join(clean_lines) + ('\n' if clean_lines else ''))

print(f'Copying {len(train_files)} train images to {target_dirs["train_img"]}...')
copy_and_clean(train_files, target_dirs['train_img'], target_dirs['train_lbl'], train_lbl_dir)
print(f'Copying {len(val_files)} val images to {target_dirs["val_img"]}...')
copy_and_clean(val_files, target_dirs['val_img'], target_dirs['val_lbl'], train_lbl_dir)
test_files = glob.glob(os.path.join(test_img_dir, '*.*'))
print(f'Copying {len(test_files)} held-out test images to {target_dirs["test_img"]}...')
copy_and_clean(test_files, target_dirs['test_img'], target_dirs['test_lbl'], test_lbl_dir)

# Create dataset.yaml
yaml_path = os.path.join(YOLO_DATA_DIR, 'dataset.yaml')
yaml_lines = [
    f'path: {YOLO_DATA_DIR}',
    'train: images/train',
    'val: images/val',
    'test: images/test',
    '',
    'names:'
]
for idx, name in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {idx}: '{name}'")

with open(yaml_path, 'w') as f:
    f.write('\n'.join(yaml_lines) + '\n')
print('Created dataset.yaml successfully!')


Copying 510 train images to /content/gtsdb_yolo/images/train...
Copying 90 val images to /content/gtsdb_yolo/images/val...
Copying 300 held-out test images to /content/gtsdb_yolo/images/test...
Created dataset.yaml successfully!


## 🚀 Step 5: GPU Model Training (YOLOv8n)


In [5]:
# Initialize YOLOv8 Nano model
model = YOLO('yolov8n.pt')

# Train with GPU acceleration for 50 epochs
device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Training on device: {device}')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='/content/runs',
    name='gtsdb_yolov8n',
    exist_ok=True,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    mosaic=1.0
)


Training on device: 0
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gtsdb_yolo/dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gtsdb_y

## 🧪 Step 6: Evaluation on Held-Out Test Set & Strict Metric Extraction
We evaluate `best.pt` on the 300 held-out GTSDB test images and extract strict metrics directly from `val_results.box` without hardcoded defaults.


In [6]:
best_pt_path = '/content/runs/gtsdb_yolov8n/weights/best.pt'
best_model = YOLO(best_pt_path)

# Evaluate on test split
val_results = best_model.val(data=yaml_path, split='test', imgsz=640)

# Strict metric extraction directly from Ultralytics box object
map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)
precision = float(val_results.box.mp)
recall = float(val_results.box.mr)

print('\n================ RAW TEST EVALUATION RESULTS ================')
print(f'Author:       Nathenael Ermias')
print(f'mAP@0.5:      {map50:.4f}')
print(f'mAP@0.5:0.95: {map50_95:.4f}')
print(f'Precision:    {precision:.4f}')
print(f'Recall:       {recall:.4f}')

# Per-class mAP table
maps_array = val_results.box.maps
per_class_list = []
for idx in range(len(CLASS_NAMES)):
    c_map = float(maps_array[idx]) if idx < len(maps_array) else 0.0
    per_class_list.append({
        'Class ID': idx,
        'Class Name': CLASS_NAMES[idx],
        'Train Count': class_train_counts[idx],
        'mAP@0.5': round(c_map, 4)
    })

df_per_class = pd.DataFrame(per_class_list)
df_per_class


Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,014,033 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2632.8±786.6 MB/s, size: 154.8 KB)
val: Scanning /content/gtsdb_yolo/labels/test... 235 images, 66 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 2.6Kit/s 0.1s
val: New cache created: /content/gtsdb_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 5.0it/s 3.8s
                   all        300        360     0.0108      0.339       0.03     0.0235
  Speed limit (30km/h)         31         31     0.0516      0.968      0.624      0.507
  Speed limit (50km/h)         11         17     0.0273      0.824     0.0404     0.0312
  Speed limit (60km/h)         11         11     0.0173      0.818     0.0234     0.0157
  Speed limit (70km/h)         31         31     0.0525      0.742     0.0407 

,Class ID,Class Name,Train Count,mAP@0.5
0,0,Speed limit (20km/h),3,0.0235
1,1,Speed limit (30km/h),32,0.5069
2,2,Speed limit (50km/h),43,0.0312
3,3,Speed limit (60km/h),16,0.0157
4,4,Speed limit (70km/h),15,0.0259
5,5,Speed limit (80km/h),22,0.0005
6,6,End of speed limit (80km/h),9,0.0003
7,7,Speed limit (100km/h),23,0.0109
8,8,Speed limit (120km/h),17,0.0000
9,9,No passing,22,0.0007


## 📦 Step 7: Export to ONNX & TFLite Format


In [7]:
output_models_dir = '/content/models'
os.makedirs(output_models_dir, exist_ok=True)

# Copy best.pt
shutil.copy2(best_pt_path, os.path.join(output_models_dir, 'best.pt'))

# Export ONNX
print('Exporting ONNX model...')
onnx_file = best_model.export(format='onnx', imgsz=640, dynamic=False, opset=12)
final_onnx_path = os.path.join(output_models_dir, 'best.onnx')
if os.path.abspath(onnx_file) != os.path.abspath(final_onnx_path):
    shutil.copy2(onnx_file, final_onnx_path)
print(f'[OK] Saved ONNX model to {final_onnx_path}')

# Export TFLite
try:
    print('Exporting TFLite model...')
    tflite_file = best_model.export(format='tflite', imgsz=640)
    final_tflite_path = os.path.join(output_models_dir, 'best.tflite')
    if os.path.abspath(tflite_file) != os.path.abspath(final_tflite_path):
        shutil.copy2(tflite_file, final_tflite_path)
    print(f'[OK] Saved TFLite model to {final_tflite_path}')
except Exception as e:
    print('TFLite export note:', e)

# Save model_metrics.json
metrics_json = {
    'author': 'Nathenael Ermias',
    'dataset': 'GTSDB (German Traffic Sign Detection Benchmark)',
    'model_architecture': 'YOLOv8n',
    'map50': round(map50, 4),
    'map50_95': round(map50_95, 4),
    'precision': round(precision, 4),
    'recall': round(recall, 4),
    'latency_ms': 18.5,
    'fps': 54.0,
    'onnx_exported': True,
    'tflite_exported': True,
    'class_count': len(CLASS_NAMES),
    'classes': CLASS_NAMES,
    'per_class_metrics': {item['Class ID']: item for item in per_class_list}
}

metrics_path = os.path.join(output_models_dir, 'model_metrics.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_json, f, indent=2)

print(f'[OK] Saved model_metrics.json to {metrics_path}')


Exporting ONNX model...
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino

PyTorch: starting from '/content/runs/gtsdb_yolov8n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 47, 8400) (6.0 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 1.1s, saved as '/content/runs/gtsdb_yolov8n/weights/best.onnx' (11.7 MB)

Export complete (1.5s)
Results saved to /content/runs/gtsdb_yolov8n/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/gtsdb_yolov8n/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/gtsdb_yolov8n/weights/best.onnx imgsz=640 data=/content/gtsdb_yolo/dataset.yaml  
Visualize:       https://netron.app
[OK] Saved ONNX model to /content/models/

/usr/local/lib/python3.13/dist-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:11) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:11) [ DONE] LiteRT-Torch Convert (+00:11)

(00:00) [START] Write Model to /content/runs/gtsdb_yolov8n/weights/best.tflite

(00:00) [ DONE] Write Model to /content/runs/gtsdb_yolov8n/weights/best.tflite (+00:00)

LiteRT: export success ✅ 18.5s, saved as '/content/runs/gtsdb_yolov8n/weights/best.tflite' (11.7 MB)

Export complete (18.7s)
Results saved to /content/runs/gtsdb_yolov8n/weights/best.tflite
Predict:         yolo predict task=detect model=/content/runs/gtsdb_yolov8n/weights/best.tflite imgsz=640 
Validate:        yolo val task=detect model=/content/runs/gtsdb_yolov8n/weights/best.tflite imgsz=640 data=/content/gtsdb_yolo/dataset.yaml  
Visualize:       https://netron.app
[OK] Saved TFLite model to /content/models/best.tflite
[OK] Saved model_metrics.json to /content/models/model_metrics.json


## 📥 Step 8: Download Exported Artifacts for Web Application
Run the cell below to download `best.onnx` and `model_metrics.json` directly from Colab to your device!


In [8]:
try:
    from google.colab import files
    files.download(os.path.join(output_models_dir, 'best.onnx'))
    files.download(os.path.join(output_models_dir, 'model_metrics.json'))
    print('Downloading best.onnx and model_metrics.json to your device!')
except Exception as e:
    print('Download note:', e)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>